In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# 1) Parámetros del modelo
# ==============================

m  = 800.0      # kg
Iy = 1100.0     # kg·m^2

# Geometría: batalla y CG descentrado
L = 3.6         # m (distancia entre ejes)
a = 2.0         # m (CG -> eje delantero)
b = 1.6         # m (CG -> eje trasero)

# Brazos aero (respecto al CG)
D_fw    = a + 0.75   # alerón delantero 0.75 m delante del eje delantero
D_rw    = b + 0.40   # alerón trasero 0.4 m detrás del eje trasero
D_floor = 0.2       # centro de presión del piso 0.3 m detrás del CG

# Suspensión
Kf = 70000.0
Kr = 70000.0
Cf = 1
Cr = 1
# Altura de referencia
L0 = 0.35
g  = 9.81
h_ref = L0 - (m * g ) / (Kf + Kr)
h0 = h_ref

# Aerodinámica
C_floor = 1.3
h_stall = 0.1
delta_h = 0.001
h_min   = 0.00005

C_fw = 1.3
C_rw = 1.4

K_fw = 0.5
K_rw = 0.7

# Velocidad
v_max = 330.0 / 3.6
tau_v = 5

# ==============================
# 2) Funciones auxiliares
# ==============================

def velocidad(t):

    return v_max * (1.0 - np.exp(-t / tau_v))

def floor_factor(h):
   h_eff = max(h, h_min)
   stall_term = 1.0 / (1.0 + np.exp(((h_stall - h_eff) / delta_h))**1.8)
   return max(((h_ref / h_eff)**1.5) * stall_term, 0.01)

def floor_downforce(v, h):
  q = v**2
  return C_floor * q * floor_factor(h)

def front_wing_downforce(v, theta):
    q = v**2
    factor = 1.0 + K_fw * theta
    return C_fw * q * factor

def rear_wing_downforce(v, theta):
    q = v**2
    factor = 1.0 + K_rw * theta
    return C_rw * q * factor

# ==============================
# 3) RHS del sistema
# ==============================

def rhs(t, x):
    z, zdot, theta, thetadot = x
    v = velocidad(t)

    z_front = z - a * theta
    z_rear  = z + b * theta

    x_f = z_front - L0
    x_r = z_rear  - L0
    xfdot = zdot - a * thetadot
    xrdot = zdot - b * thetadot

    Ff = -Kf * x_f - Cf * xfdot
    Fr = -Kr * x_r - Cr * xrdot

    h = z
    F_floor = floor_downforce(v, h)
    F_fw    = front_wing_downforce(v, theta)
    F_rw    = rear_wing_downforce(v, theta)
    F_aero_total = F_floor + F_fw + F_rw

    zddot = (Ff + Fr - F_aero_total - m * g) / m
    M_susp = -a * Ff + b * Fr
    M_aero = (-D_fw)*(-F_fw) + (D_rw)*(-F_rw) + (D_floor)*(-F_floor)
    thetaddot = (M_susp + M_aero) / Iy

    return np.array([zdot, zddot, thetadot, thetaddot])

# ==============================
# 4) Integración RK4
# ==============================

def rk4_step(f, t, x, dt):
    k1 = f(t, x)
    k2 = f(t + 0.5*dt, x + 0.5*dt*k1)
    k3 = f(t + 0.5*dt, x + 0.5*dt*k2)
    k4 = f(t + dt, x + dt*k3)
    return x + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

# ==============================
# 5) Simulación
# ==============================
t0, tf, dt = 0.0, 20.0, 0.07
N = int((tf - t0)/dt) + 1
#xr0 = (a*(m*g ))/(L*(Kf+Kr))
#print(int(xr0))
#xf0 = (b*(m*g ))/(L*(Kf+Kr))
#print(int(xf0))
#theta0 = (xf0-xr0)/L
#print(int(theta0))
x = np.array([h_ref, 0.0, -0.00338, 0.0]) #-0.00308
ts = np.zeros(N); zs = np.zeros(N); ths = np.zeros(N); vs = np.zeros(N)
F_floor_hist = np.zeros(N); F_fw_hist = np.zeros(N); F_rw_hist = np.zeros(N)

for i in range(N):
    t = t0 + i*dt
    ts[i] = t
    zs[i] = x[0]
    ths[i] = x[2]
    v = velocidad(t)
    vs[i] = v
    h = x[0]
    F_floor_hist[i] = floor_downforce(v, h)
    F_fw_hist[i] = front_wing_downforce(v, x[2])
    F_rw_hist[i] = rear_wing_downforce(v, x[2])
    x = rk4_step(rhs, t, x, dt)

# ==============================
# 6) Gráficas generales
# ==============================

plt.figure(); plt.plot(ts, vs*3.6)
plt.xlabel("Tiempo [s]"); plt.ylabel("Velocidad [km/h]")
plt.title("Perfil de velocidad"); plt.grid(True)
plt.figure(); plt.plot(ts, zs*1000)

plt.xlabel("Tiempo [s]"); plt.ylabel("z [mm]")
plt.title("Posición vertical CG"); plt.grid(True)
plt.xlim(0,20)
plt.figure(); plt.plot(ts, ths*180/np.pi)

plt.xlabel("Tiempo [s]"); plt.ylabel("θ [°]")
plt.title("Pitch"); plt.grid(True)
plt.xlim(0,20)
plt.figure()

plt.plot(ts, F_floor_hist/1000, label="Piso")
plt.plot(ts, F_fw_hist/1000, label="Alerón delantero")
plt.plot(ts, F_rw_hist/1000, label="Alerón trasero")

plt.xlabel("Tiempo [s]"); plt.ylabel("Fuerza [kN]")
plt.title("Fuerzas aerodinámicas totales"); plt.legend(); plt.grid(True)

plt.show()


In [ ]:
# ==============================
# 7) Aceleración con diferencias centrales
# ==============================

# z es posición del CG (desviación respecto a L0), en metros
# la aceleración será en m/s^2
acc = np.zeros_like(zs)

# Diferencias centrales para los puntos interiores
for i in range(1, N-1):
    acc[i] = (zs[i+1] - 2.0*zs[i] + zs[i-1]) / (dt**2)

# Para los extremos, usamos una aproximación sencilla (adelante/atrás)
acc[0]  = (zs[2] - 2.0*zs[1] + zs[0]) / (dt**2)
acc[-1] = (zs[-1] - 2.0*zs[-2] + zs[-3]) / (dt**2)

# Si quieres, también en "g"
acc_g = acc / g

# Gráfica de la aceleración
plt.figure()
plt.plot(ts, acc)
plt.xlabel("Tiempo [s]")
plt.ylabel("Aceleración vertical [m/s²]")
plt.title("Aceleración vertical del CG (diferencias centrales)")
plt.grid(True)

# Opcional: aceleración en g
plt.figure()
plt.plot(ts, acc_g)
plt.xlabel("Tiempo [s]")
plt.ylabel("Aceleración vertical [g]")
plt.title("Aceleración vertical del CG en múltiplos de g")
plt.xlim(15,20)
plt.grid(True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# 7) Barrido en altura para el piso a 70 m/s
# ==============================

v_plot = 70.0  # m/s (~252 km/h)
def floor_factor(h):
   h_eff = max(h, h_min)
   stall_term = 1.0 / (1.0 + np.exp(((h_stall - h_eff) / delta_h))**1)
   return ((h_ref / h_eff)**1.5) * stall_term

def floor_downforce(v, h):
  q = v**2
  return C_floor * q * floor_factor(h)
# Rango de alturas [m]
# Puedes ajustar el máximo si quieres
h_vals = np.linspace(max(h_min*1.01, 0.01), 0.50, 1000)

# OJO: tu floor_factor usa max(h, h_min) y no es vectorizada,
# así que hacemos el barrido con comprensión de listas
ff_vals = np.array([floor_factor(h) for h in h_vals])
Fd_vals = np.array([floor_downforce(v_plot, h) for h in h_vals])

plt.figure(figsize=(8, 8))

# --- 1) floor_factor(h) ---
plt.subplot(2, 1, 1)
plt.plot(h_vals*1000, ff_vals)
plt.xlabel("Altura h [mm]")
plt.ylabel("floor_factor [-]")
plt.title("Factor de piso vs altura (v = 70 m/s)")
plt.grid(True)

# --- 2) Fuerza de piso F_floor(h) ---
plt.subplot(2, 1, 2)
plt.plot(h_vals*1000, Fd_vals/1000.0)
plt.xlabel("Altura h [mm]")
plt.ylabel("F_floor [kN]")
plt.title("Fuerza de piso vs altura (v = 70 m/s)")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
def analizar_z_fft(t, z, t_min=5.0, f_max=30.0):
    """
    t     : vector de tiempo [s]
    z     : señal z(t)
    t_min : tiempo desde el cual considerar datos (para sacar transitorio)
    f_max : frecuencia máxima a mostrar [Hz]
    """
    t = np.asarray(t)
    z = np.asarray(z)

    # Filtrar desde t_min
    mask = t >= t_min
    t = t[mask]
    z = z[mask]

    if len(t) < 4:
        print("Muy pocos puntos después de t_min. Ajusta t_min o el tiempo de simulación.")
        return

    dt_local = t[1] - t[0]
    fs = 1.0 / dt_local

    # Quitar componente DC
    z_detrend = z - np.mean(z)

    # Ventana de Hann
    ventana = np.hanning(len(z_detrend))
    z_win = z_detrend * ventana

    # FFT real
    Z = np.fft.rfft(z_win)
    freqs = np.fft.rfftfreq(len(z_win), d=dt_local)
    mag = np.abs(Z) / len(z_win)

    # Limitar frecuencia máxima
    if f_max is not None:
        mask_f = freqs <= f_max
        freqs = freqs[mask_f]
        mag   = mag[mask_f]

    # Frecuencia dominante (excluyendo 0 Hz)
    if len(freqs) > 1:
        idx_max = np.argmax(mag[1:]) + 1
        f_dom = freqs[idx_max]
    else:
        f_dom = None

    # Gráficos
    fig, axs = plt.subplots(2, 1, figsize=(8, 6))

    # z(t) usado en el análisis
    axs[0].plot(t, z*1000.0)
    axs[0].set_xlabel("Tiempo [s]")
    axs[0].set_ylabel("z [mm]")
    axs[0].set_title(f"z(t) usada en FFT (t ≥ {t_min:.1f} s)")
    axs[0].grid(True)

    # Espectro
    axs[1].plot(freqs, mag)
    axs[1].set_xlabel("Frecuencia [Hz]")
    axs[1].set_ylabel("|Z(f)|")
    titulo_fft = "Espectro de z(t)"
    if f_dom is not None:
        titulo_fft += f" – f_dom ≈ {f_dom:.2f} Hz"
    axs[1].set_title(titulo_fft)
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

    if f_dom is not None:
        print(f"Frecuencia dominante ≈ {f_dom:.3f} Hz")
    else:
        print("No se pudo determinar una frecuencia dominante clara.")

# Llamada al análisis (ajusta t_min y f_max si quieres)
analizar_z_fft(ts, zs, t_min=5.4, f_max=20.0)

In [ ]:
def calcular_amplitud_promedio_z(t, z, t_min=7.5):
    """
    Calcula amplitud promedio de la oscilación de z(t) en estado casi estacionario.

    t     : vector de tiempo [s]
    z     : z(t)
    t_min : tiempo desde el que se considera la señal (se elimina transitorio)
    """
    t = np.asarray(t)
    z = np.asarray(z)

    mask = t >= t_min
    t_ss = t[mask]
    z_ss = z[mask]

    if len(t_ss) < 4:
        print("Muy pocos puntos después de t_min. Ajusta t_min o el tiempo de simulación.")
        return None, None

    # Quitamos nivel medio
    z_detrend = z_ss - np.mean(z_ss)

    # Amplitud pico a pico / 2 (semi-amplitud)
    amp_pp = np.max(z_detrend) - np.min(z_detrend)
    amp_med = 0.5 * amp_pp

    # Amplitud RMS
    amp_rms = np.sqrt(np.mean(z_detrend**2))

    print(f"Amplitud media (pico a pico/2) ≈ {amp_med*1000:.3f} mm")
    print(f"Amplitud RMS ≈ {amp_rms*1000:.3f} mm")

    return amp_med, amp_rms



In [ ]:


# Cálculo de amplitud promedio
amp_med, amp_rms = calcular_amplitud_promedio_z(ts, zs, t_min=5)


In [ ]:
def analizar_theta_fft(t, theta, t_min=7.5, f_max=30.0):
    """
    Análisis en frecuencia del pitch θ(t).

    t      : vector de tiempo [s]
    theta  : señal θ(t) [rad]
    t_min  : tiempo desde el que se considera la señal (para eliminar transitorio)
    f_max  : frecuencia máxima a mostrar [Hz]
    """
    t = np.asarray(t)
    theta = np.asarray(theta)

    # Filtrar desde t >= t_min
    mask = t >= t_min
    t = t[mask]
    theta = theta[mask]

    if t.size < 4:
        print("Muy pocos puntos después de t_min. Ajusta t_min o el tiempo de simulación.")
        return

    # Paso de tiempo
    dt = t[1] - t[0]

    # Quitar componente DC
    theta_detrend = theta - np.mean(theta)

    # Ventana de Hann
    ventana = np.hanning(theta_detrend.size)
    theta_win = theta_detrend * ventana

    # FFT real
    TH = np.fft.rfft(theta_win)
    freqs = np.fft.rfftfreq(theta_win.size, d=dt)
    mag = np.abs(TH) / theta_win.size

    # Limitar frecuencia máxima
    if f_max is not None:
        mask_f = freqs <= f_max
        freqs = freqs[mask_f]
        mag   = mag[mask_f]

    # Frecuencia dominante (excluyendo 0 Hz)
    if freqs.size > 1:
        idx_max = 1 + np.argmax(mag[1:])
        f_dom = freqs[idx_max]
    else:
        f_dom = None

    # ==========================
    # Gráficos
    # ==========================
    fig, axs = plt.subplots(2, 1, figsize=(8, 6))

    # 1) θ(t) en el tiempo (en grados)
    axs[0].plot(t, theta * 180.0 / np.pi)
    axs[0].set_xlabel("Tiempo [s]")
    axs[0].set_ylabel("Pitch θ [°]")
    axs[0].set_title(f"Pitch θ(t) usado en FFT (t ≥ {t_min:.1f} s)")
    axs[0].grid(True)

    # 2) Espectro de frecuencias
    axs[1].plot(freqs, mag)
    axs[1].set_xlabel("Frecuencia [Hz]")
    axs[1].set_ylabel("|Θ(f)|")
    if f_dom is not None:
        axs[1].set_title(f"Espectro de θ(t)")
    else:
        axs[1].set_title("Espectro de θ(t)")
    axs[1].grid(True)

    plt.tight_layout()
    plt.show()

    if f_dom is not None:
        print(f"Frecuencia dominante del pitch")
    else:
        print("No se pudo determinar una frecuencia dominante clara para el pitch.")


In [ ]:
analizar_theta_fft(ts, ths, t_min=5, f_max=20.0)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# Parámetros (MODIFICA AQUÍ)
# ==========================
h_min   = 0.010   # m
h_ref   = 0.15   # m
h_stall = 0.1   # m
delta_h = 0.001   # m

C_floor = 3.0     # "Cz_floor" efectivo (ajústalo a tu modelo)
v_plot  = 70.0    # m/s, velocidad a la que quieres ver el downforce vs h

# ==========================
# Definición de funciones
# ==========================

def floor_factor(h):
    """
    h: altura piso-suelo [m] (puede ser float o array de numpy)
    """
    # Para poder usar arrays: np.maximum en vez de max
    h_eff = np.maximum(h, h_min)
    stall_term = 1.0 / (1.0 + np.exp(((h_stall - h_eff) / delta_h))**2)
    return ((h_ref / h_eff)**1.2) * stall_term

def floor_downforce(v, h):
    """
    v: velocidad [m/s]
    h: altura piso-suelo [m] (float o array)
    """
    q = v**2  # tú definiste q así; si quieres, cambia a 0.5*rho*v**2
    return C_floor * q * floor_factor(h)

# ==========================
# Rango para graficar
# ==========================
h_vals = np.linspace(0.085, 0.120, 400)  # 5 mm a 160 mm

ff_vals   = floor_factor(h_vals)
fd_vals_h = floor_downforce(v_plot, h_vals)

# ==========================
# Gráficos
# ==========================

plt.figure(figsize=(8, 6))


# 2) floor_downforce vs h a velocidad fija
plt.subplot(2, 1, 2)
plt.plot(h_vals * 1000.0, fd_vals_h)
plt.xlabel("Altura piso-suelo h [mm]")
plt.ylabel("floor_downforce(v, h) [unid. arbitrarias]")
plt.title(f"Floor downforce vs altura")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# -*- coding: utf-8 -*-
"""Modelo académico heave/pitch integrado exclusivamente con RK4.

Los valores predeterminados son ilustrativos, NO un reglaje identificado de F1.
Convención: z hacia arriba; theta positivo baja el frente y sube la parte trasera.
k y c representan un eje completo. Apoyos fijos, pista plana, ángulos pequeños.
Ver LEEME.md para ecuaciones, fuentes y experimentos propuestos.
"""
from __future__ import annotations

import argparse
import csv
import json
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import root
from scipy.signal import detrend
from scipy.special import expit


# 1. Un solo conjunto de parámetros y unidades explícitas.
@dataclass(frozen=True)
class Parametros:
    m_kg: float = 800.0
    iy_kg_m2: float = 1100.0
    a_m: float = 2.0
    b_m: float = 1.6
    g_m_s2: float = 9.81
    kf_n_m: float = 70000.0
    kr_n_m: float = 70000.0
    l0f_m: float = 0.35
    l0r_m: float = 0.35
    cf_comp_ns_m: float = 3000.0
    cf_ext_ns_m: float = 5000.0
    cr_comp_ns_m: float = 3000.0
    cr_ext_ns_m: float = 5000.0
    # 0: cambio por ramas exacto; >0: transición suave para estudiar regularidad.
    suavizado_amortiguador_m_s: float = 0.005
    rho_kg_m3: float = 1.225
    # Producto A*C_L [m²], C_L tomado como magnitud positiva de carga hacia abajo.
    cla_piso_m2: float = 2.122448979591837
    cla_frente_m2: float = 2.122448979591837
    cla_trasero_m2: float = 2.2857142857142856
    d_frente_m: float = 2.75
    d_trasero_m: float = 2.0
    d_piso_m: float = 0.2  # positivo detrás del CG
    sensibilidad_frente_rad_inv: float = 0.5
    sensibilidad_trasero_rad_inv: float = 0.7
    # Separar altura CG y piso; 0 conserva la idealización del archivo inicial.
    cg_a_piso_m: float = 0.0
    h_aero_ref_m: float = 0.294
    h_stall_m: float = 0.10
    ancho_stall_m: float = 0.001
    exponente_piso: float = 1.5
    factor_piso_min: float = 0.01
    h_guardia_m: float = 0.00005
    h_contacto_m: float = 0.001
    # 0: cuatro estados. >0: quinto estado, carga de piso con relajación temporal.
    tau_aero_s: float = 0.0

    @property
    def batalla_m(self):
        return self.a_m + self.b_m

    def validar(self):
        positivos = (self.m_kg, self.iy_kg_m2, self.a_m, self.b_m,
                     self.kf_n_m, self.kr_n_m, self.g_m_s2, self.rho_kg_m3,
                     self.h_aero_ref_m, self.ancho_stall_m, self.h_guardia_m)
        if not all(np.isfinite(v) and v > 0 for v in positivos):
            raise ValueError("Masa, inercia, geometría, rigidez y escalas deben ser positivas.")
        no_negativos = (self.cf_comp_ns_m, self.cf_ext_ns_m, self.cr_comp_ns_m,
                        self.cr_ext_ns_m, self.suavizado_amortiguador_m_s,
                        self.tau_aero_s, self.cla_piso_m2, self.cla_frente_m2,
                        self.cla_trasero_m2, self.factor_piso_min)
        if not all(np.isfinite(v) and v >= 0 for v in no_negativos):
            raise ValueError("Amortiguamiento, cargas y escalas opcionales deben ser no negativos.")
        if self.h_contacto_m <= self.h_guardia_m:
            raise ValueError("h_contacto debe superar la protección del denominador aerodinámico.")
        if not 0 <= self.factor_piso_min < 1 or self.exponente_piso <= 0:
            raise ValueError("Factor residual del piso en [0,1) y exponente positivo.")


@dataclass(frozen=True)
class Simulacion:
    tf_s: float = 30.0
    dt_s: float = 0.005
    perfil: str = "rampa"
    v_final_kmh: float = 330.0
    tau_velocidad_s: float = 5.0
    analizar_desde_s: float = 20.0
    perturbacion_z_m: float = 0.001
    perturbacion_theta_rad: float = 0.0002

    def validar(self):
        if self.tf_s <= 0 or self.dt_s <= 0 or self.v_final_kmh < 0:
            raise ValueError("Tiempo y paso positivos; velocidad no negativa.")
        if not 0 <= self.analizar_desde_s < self.tf_s:
            raise ValueError("La ventana de análisis debe estar dentro de la simulación.")
        if self.perfil not in ("rampa", "constante") or self.tau_velocidad_s <= 0:
            raise ValueError("Perfil inválido o tau de velocidad no positivo.")


# 2. Cinemática y constitutivas; estas mismas funciones alimentan TODAS las gráficas.
def velocidad(t, s):
    vmax = s.v_final_kmh / 3.6
    return vmax if s.perfil == "constante" else vmax * (-np.expm1(-t / s.tau_velocidad_s))


def cinematica(x, p):
    z, dz, theta, dtheta = x[:4]
    return z - p.a_m * theta, z + p.b_m * theta, dz - p.a_m * dtheta, dz + p.b_m * dtheta


def coef_amortiguador(u, c_comp, c_ext, epsilon=0.0):
    """Compresión u<0, extensión u>0; u es la velocidad LOCAL relativa."""
    u = np.asarray(u)
    if epsilon == 0:
        return np.where(u < 0.0, c_comp, c_ext)
    peso_ext = 0.5 * (1.0 + np.tanh(u / epsilon))
    return c_comp + (c_ext - c_comp) * peso_ext


def factor_piso(h, p):
    heff = np.maximum(h, p.h_guardia_m)
    stall = expit((heff - p.h_stall_m) / p.ancho_stall_m)
    # Mezcla suave: evita la esquina de max(factor, factor_min) del original.
    bruto = p.factor_piso_min + (1-p.factor_piso_min)*(p.h_aero_ref_m/heff)**p.exponente_piso*stall
    normalizador = p.factor_piso_min + (1-p.factor_piso_min)*float(expit((p.h_aero_ref_m-p.h_stall_m)/p.ancho_stall_m))
    return bruto / normalizador


def cargas_estaticas(v, x, p):
    h = x[0] - p.cg_a_piso_m + p.d_piso_m * x[2]
    q = 0.5 * p.rho_kg_m3 * v * v
    piso = q * p.cla_piso_m2 * factor_piso(h, p)
    frente = q * p.cla_frente_m2 * max(0.0, 1.0 + p.sensibilidad_frente_rad_inv * x[2])
    trasero = q * p.cla_trasero_m2 * max(0.0, 1.0 + p.sensibilidad_trasero_rad_inv * x[2])
    return np.array([piso, frente, trasero], dtype=float)


def fuerzas(x, v, p):
    zf, zr, uf, ur = cinematica(x, p)
    cf = float(coef_amortiguador(uf, p.cf_comp_ns_m, p.cf_ext_ns_m,
                                p.suavizado_amortiguador_m_s))
    cr = float(coef_amortiguador(ur, p.cr_comp_ns_m, p.cr_ext_ns_m,
                                p.suavizado_amortiguador_m_s))
    fdf, fdr = -cf * uf, -cr * ur
    fsf = -p.kf_n_m * (zf - p.l0f_m)
    fsr = -p.kr_n_m * (zr - p.l0r_m)
    aero = cargas_estaticas(v, x, p)
    piso_qs = aero[0]
    if p.tau_aero_s > 0:
        aero[0] = x[4]
    piso, frente, trasero = aero
    total_z = fsf + fsr + fdf + fdr - aero.sum() - p.m_kg * p.g_m_s2
    momento = -p.a_m * (fsf + fdf) + p.b_m * (fsr + fdr)
    momento += p.d_frente_m * frente - p.d_trasero_m * trasero - p.d_piso_m * piso
    potencia_d = fdf * uf + fdr * ur
    velocidades_aero = np.array([x[1] + p.d_piso_m * x[3],
                                 x[1] - p.d_frente_m * x[3],
                                 x[1] + p.d_trasero_m * x[3]])
    potencia_a = float(-aero @ velocidades_aero)
    return dict(zf=zf, zr=zr, uf=uf, ur=ur, cf=cf, cr=cr, fsf=fsf, fsr=fsr,
                fdf=fdf, fdr=fdr, piso=piso, piso_qs=piso_qs,
                frente=frente, trasero=trasero, total_z=total_z,
                momento=momento, potencia_d=potencia_d, potencia_a=potencia_a)


def rhs_velocidad(x, v, p):
    f = fuerzas(x, v, p)
    dx = [x[1], f["total_z"] / p.m_kg, x[3], f["momento"] / p.iy_kg_m2]
    if p.tau_aero_s > 0:
        dx.append((f["piso_qs"] - x[4]) / p.tau_aero_s)
    return np.asarray(dx)


def rhs(t, x, p, s):
    return rhs_velocidad(x, velocidad(t, s), p)


def holgura_minima(x, p):
    # Extremos idealizados del piso, no solo altura en el CG/centro de presión.
    base = x[0] - p.cg_a_piso_m
    return min(base - p.a_m * x[2], base + p.b_m * x[2], base + p.d_piso_m * x[2])


# 3. Equilibrio: fórmula exacta sin aero y resolución no lineal a velocidad fija.
def equilibrio_sin_aero(p):
    ff = p.m_kg * p.g_m_s2 * p.b_m / p.batalla_m
    fr = p.m_kg * p.g_m_s2 * p.a_m / p.batalla_m
    zf, zr = p.l0f_m - ff / p.kf_n_m, p.l0r_m - fr / p.kr_n_m
    theta = (zr - zf) / p.batalla_m
    z = (p.b_m * zf + p.a_m * zr) / p.batalla_m
    return np.array([z, 0.0, theta, 0.0])


def estado_con_piso(x4, v, p):
    return np.r_[x4, cargas_estaticas(v, x4, p)[0]] if p.tau_aero_s > 0 else x4.copy()


def equilibrio(p, v, guess=None, registro=None):
    """Continuación en V y multistart de respaldo. No enumera todas las raíces."""
    x4 = equilibrio_sin_aero(p) if guess is None else np.asarray(guess[:4]).copy()
    speeds = np.linspace(0.0, v, max(2, int(np.ceil(v / 2.0)) + 1)) if guess is None else [v]
    for vi in speeds:
        def residual(q):
            trial = estado_con_piso(np.array([q[0], 0.0, q[1], 0.0]), vi, p)
            f = fuerzas(trial, vi, p)
            return [f["total_z"] / (p.m_kg * p.g_m_s2),
                    f["momento"] / (p.m_kg * p.g_m_s2 * p.batalla_m)]
        sol = root(residual, x4[[0, 2]], method="hybr", tol=1e-10)
        def admisible(q):
            trial = np.array([q[0],0.0,q[1],0.0])
            return (np.all(np.isfinite(q)) and np.linalg.norm(residual(q),ord=np.inf)<1e-7
                    and holgura_minima(trial,p)>p.h_contacto_m and abs(q[1])<=0.2)
        if admisible(sol.x):
            elegido = sol.x
        else:
            # Una rama puede terminar en un pliegue. Buscar otra raíz no equivale
            # a decir que el equilibrio sea único ni a simular el salto dinámico.
            candidatas = []
            alturas = [p.h_stall_m+d for d in (-0.01,-0.002,0.0,0.002,0.01,0.03)]
            alturas.append(p.h_aero_ref_m)
            for theta0 in (x4[2],0.0,-0.015):
                for h0 in alturas:
                    z0 = h0+p.cg_a_piso_m-p.d_piso_m*theta0
                    trial = root(residual,[z0,theta0],method="hybr",tol=1e-10)
                    if admisible(trial.x) and not any(np.linalg.norm(trial.x-q)<1e-6 for q in candidatas):
                        candidatas.append(trial.x)
            if not candidatas:
                raise ValueError(f"Sin raíz admisible encontrada a V={vi*3.6:.1f} km/h; revisar parámetros/semillas.")
            previo = x4[[0,2]].copy()
            def distancia_alturas(q):
                dz,dtheta = q-previo
                return np.hypot(dz-p.a_m*dtheta,dz+p.b_m*dtheta)
            elegido = min(candidatas,key=distancia_alturas)
            if registro is not None:
                registro.append(dict(v_kmh=float(vi*3.6),raices_admisibles_encontradas=len(candidatas),
                                     nota="La continuación necesitó multistart; posible cambio de rama, no búsqueda exhaustiva."))
        x4[[0, 2]] = elegido
    return estado_con_piso(x4, v, p)


# 4. Runge–Kutta clásico de cuarto orden, con paso fijo.
def paso_rk4(f, t, x, dt):
    k1 = f(t, x)
    k2 = f(t + dt / 2, x + dt * k1 / 2)
    k3 = f(t + dt / 2, x + dt * k2 / 2)
    k4 = f(t + dt, x + dt * k3)
    return x + dt * (k1 + 2*k2 + 2*k3 + k4) / 6


def integrar(p, s, x0=None):
    p.validar()
    s.validar()
    if p.tau_aero_s > 0 and s.dt_s/p.tau_aero_s >= 2.7:
        raise ValueError("Retardo aero rápido: reducir dt. Como inicio, usar dt <= tau_aero/10.")
    registro_equilibrio = []
    if x0 is None:
        v0 = float(velocidad(0.0, s))
        x0 = equilibrio(p, v0,registro=registro_equilibrio)
        x0[0] += s.perturbacion_z_m
        x0[2] += s.perturbacion_theta_rad
        # Estado aero inicialmente ajustado a la geometría perturbada.
        if p.tau_aero_s > 0:
            x0[4] = cargas_estaticas(v0, x0, p)[0]
    if len(x0) != (5 if p.tau_aero_s > 0 else 4) or not np.all(np.isfinite(x0)):
        raise ValueError("Dimensión o valores inválidos de la condición inicial.")
    if holgura_minima(x0, p) <= p.h_contacto_m:
        raise ValueError("La condición inicial está fuera del dominio sin contacto.")
    if abs(x0[2]) > 0.2:
        raise ValueError("Ángulo inicial fuera de la aproximación pequeña.")
    n = max(3, int(np.ceil(s.tf_s / s.dt_s - 1e-10)))
    ts = np.linspace(0.0, s.tf_s, n + 1)
    dt = ts[1] - ts[0]  # malla uniforme con tf exacto
    nfev = 0
    class FueraDeDominio(Exception):
        pass
    def f(t, x):
        nonlocal nfev
        if holgura_minima(x,p) <= p.h_contacto_m:
            raise FueraDeDominio("umbral de contacto en una etapa RK4")
        if abs(x[2]) > 0.2:
            raise FueraDeDominio("ángulo fuera de la aproximación pequeña (|theta|>0.2 rad)")
        nfev += 1
        return rhs(t,x,p,s)
    estado = "completa"
    xs = np.empty((len(ts), len(x0)))
    xs[0] = x0
    for i in range(n):
        try:
            trial = paso_rk4(f, ts[i], xs[i], dt)
        except FueraDeDominio as exc:
            estado = f"{exc}, entre t={ts[i]:.6g} y {ts[i+1]:.6g} s; refinamiento necesario"
            xs, ts = xs[:i+1], ts[:i+1]
            break
        if not np.all(np.isfinite(trial)):
            estado = "integración no finita; revisar estabilidad/paso"
            xs, ts = xs[:i+1], ts[:i+1]
            break
        if holgura_minima(trial, p) <= p.h_contacto_m:
            estado = f"umbral de contacto entre t={ts[i]:.6g} y {ts[i+1]:.6g} s; refinamiento necesario"
            xs, ts = xs[:i+1], ts[:i+1]
            break
        if abs(trial[2]) > 0.2:
            estado = f"ángulo final fuera de la aproximación pequeña entre t={ts[i]:.6g} y {ts[i+1]:.6g} s"
            xs, ts = xs[:i+1], ts[:i+1]
            break
        xs[i+1] = trial
    return dict(t=ts, x=xs, metodo="RK4", dt=dt, estado=estado, nfev=nfev,
                registro_equilibrio=registro_equilibrio)


# 5. Diferencias finitas, FFT normalizada y balance de energía.
def segunda_derivada(y, dt):
    if len(y) < 4:
        return np.full(len(y), np.nan)
    out = np.empty_like(y)
    out[1:-1] = (y[2:] - 2*y[1:-1] + y[:-2]) / dt**2
    out[0] = (2*y[0] - 5*y[1] + 4*y[2] - y[3]) / dt**2
    out[-1] = (2*y[-1] - 5*y[-2] + 4*y[-3] - y[-4]) / dt**2
    return out


def espectro(t, signal):
    if len(t) < 8:
        raise ValueError("Se necesitan al menos ocho muestras para analizar la FFT.")
    dt = t[1] - t[0]
    if not np.allclose(np.diff(t), dt):
        raise ValueError("La FFT requiere muestreo uniforme.")
    y = detrend(signal, type="linear")
    w = np.hanning(len(y))
    f = np.fft.rfftfreq(len(y), dt)
    amp = np.abs(np.fft.rfft(y*w)) / w.sum()
    amp[1:] *= 2
    if len(y) % 2 == 0:
        amp[-1] /= 2  # Nyquist no se duplica
    idx = int(np.argmax(amp[1:]) + 1)
    dominante = float(f[idx]) if np.ptp(y) > 1e-12 else None
    return dict(f=f, amplitud=amp, dominante_hz=dominante,
                resolucion_hz=1 / (len(t)*dt), nyquist_hz=1 / (2*dt))


def diagnosticos(r, p, s):
    ts, xs = r["t"], r["x"]
    vs = np.asarray([velocidad(t, s) for t in ts])
    fs = [fuerzas(x, v, p) for x, v in zip(xs, vs)]
    data = {k: np.asarray([f[k] for f in fs]) for k in fs[0]}
    data["velocidad"] = vs
    data["acc_rhs"] = data["total_z"] / p.m_kg
    data["acc_df"] = segunda_derivada(xs[:, 0], r["dt"])
    energia = (0.5*p.m_kg*xs[:, 1]**2 + 0.5*p.iy_kg_m2*xs[:, 3]**2
               + 0.5*p.kf_n_m*(data["zf"]-p.l0f_m)**2
               + 0.5*p.kr_n_m*(data["zr"]-p.l0r_m)**2 + p.m_kg*p.g_m_s2*xs[:, 0])
    trabajo = cumulative_trapezoid(data["potencia_d"] + data["potencia_a"], ts, initial=0.0)
    data["energia_mecanica"] = energia
    data["error_energia"] = energia - energia[0] - trabajo
    mask = ts >= s.analizar_desde_s
    notas = [f"Inicialización: multistart a {item['v_kmh']:.1f} km/h; posible cambio de rama de equilibrio."
             for item in r.get("registro_equilibrio",[])]
    if mask.sum() < 8:
        return data, {"estado": r["estado"], "notas": ["Sin ventana final suficiente para FFT."]}
    t, z, th = ts[mask], xs[mask, 0], xs[mask, 2]
    ez, et = espectro(t, z), espectro(t, th)
    z_d = detrend(z, type="linear")
    cambio_v = float(np.ptp(vs[mask]) / max(np.mean(vs[mask]), 1e-9))
    mitad = len(z_d)//2
    rms1 = float(np.sqrt(np.mean(z_d[:mitad]**2)))
    rms2 = float(np.sqrt(np.mean(z_d[mitad:]**2)))
    if cambio_v > 0.01:
        notas.append("V cambia más de 1% en la ventana: FFT describe un tramo no estacionario.")
    if abs(rms2-rms1) > 0.1 * max(rms1, rms2, 1e-12):
        notas.append("RMS entre mitades cambia más de 10%; revisar transitorio/envolvente.")
    if r["estado"] != "completa":
        notas.append("Integración incompleta: no interpretar como régimen final.")
    # Esto es una prueba de consistencia, no una validación del vehículo real.
    residual_est = fuerzas(equilibrio(p, float(velocidad(0, s))), float(velocidad(0, s)), p)
    summary = dict(estado=r["estado"], metodo=r["metodo"], dt_real_s=r["dt"], nfev=r["nfev"],
                   t_final_s=float(ts[-1]), f_z_hz=ez["dominante_hz"], f_theta_hz=et["dominante_hz"],
                   resolucion_fft_hz=ez["resolucion_hz"], nyquist_hz=ez["nyquist_hz"],
                   semiamplitud_z_mm=float(np.ptp(z_d)*500), rms_z_mm=float(np.sqrt(np.mean(z_d**2))*1000),
                   cambio_velocidad_relativo=cambio_v, rms_primera_mitad_mm=rms1*1000,
                   rms_segunda_mitad_mm=rms2*1000, holgura_minima_mm=float(min(holgura_minima(x,p) for x in xs)*1000),
                   max_error_energia_j=float(np.max(np.abs(data["error_energia"]))),
                   max_error_acc_df_m_s2=float(np.nanmax(np.abs(data["acc_df"][1:-1]-data["acc_rhs"][1:-1]))),
                   max_potencia_amortiguadores_w=float(np.max(data["potencia_d"])),
                   residual_equilibrio_n=float(residual_est["total_z"]), notas=notas)
    data["espectro_z"], data["espectro_theta"] = ez, et
    return data, summary


def jacobiano(f, x):
    cols = []
    for j in range(len(x)):
        h = 1e-6 * max(1.0, abs(x[j]))
        d = np.zeros_like(x)
        d[j] = h
        cols.append((f(x+d)-f(x-d))/(2*h))
    return np.column_stack(cols)


def analisis_lineal(p, v):
    registro = []
    xeq = equilibrio(p, v,registro=registro)
    if p.suavizado_amortiguador_m_s == 0 and (p.cf_comp_ns_m != p.cf_ext_ns_m or p.cr_comp_ns_m != p.cr_ext_ns_m):
        return {"nota": "Ramas distintas: Jacobiano clásico no definido en velocidad cero; usar transición suave o análisis por ramas."}
    eig = np.linalg.eigvals(jacobiano(lambda x: rhs_velocidad(x, v, p), xeq))
    return {"equilibrio": xeq.tolist(), "autovalores": [[float(z.real),float(z.imag)] for z in eig],
            "max_parte_real_s_inv": float(np.max(eig.real)),
            "registro_equilibrio": registro,
            "nota": "Estabilidad local del modelo idealizado; no certifica un ciclo límite."}


# 6. Convergencia y Richardson: todas las soluciones se obtienen con RK4.
def estudiar_convergencia(p, s):
    n_base = max(3, int(np.ceil(s.tf_s / s.dt_s - 1e-10)))
    dt_base = s.tf_s / n_base
    casos = [replace(s,dt_s=dt_base/factor) for factor in (1,2,4,8)]
    resultados = [integrar(p,caso) for caso in casos]
    filas = []
    if any(r["estado"] != "completa" for r in resultados):
        return [dict(dt_s=r["dt"],estado=r["estado"],nota="Comparación no válida: alguna integración se interrumpió.") for r in resultados]
    # Comparar en los mismos tiempos de la malla más gruesa evita interpolación.
    comunes = [r["x"][::factor] for r,factor in zip(resultados,(1,2,4,8))]
    dz = [float(np.max(np.abs(comunes[j][:,0]-comunes[j+1][:,0]))) for j in range(3)]
    dth = [float(np.max(np.abs(comunes[j][:,2]-comunes[j+1][:,2]))) for j in range(3)]
    for j,(r,caso) in enumerate(zip(resultados,casos)):
        _, metrics = diagnosticos(r,p,caso)
        orden = float(np.log2(dz[j]/dz[j+1])) if j < 2 and min(dz[j:j+2]) > 1e-14 else None
        regimen_4 = bool(3.8<=orden<=4.2) if orden is not None else None
        fila = dict(metodo="RK4",dt_s=r["dt"],nfev=r["nfev"],estado=r["estado"],
                    diferencia_ref_rk4_dt8_z_mm=float(np.max(np.abs(comunes[j][:,0]-comunes[-1][:,0])))*1000,
                    diferencia_ref_rk4_dt8_theta_grados=float(np.degrees(np.max(np.abs(comunes[j][:,2]-comunes[-1][:,2])))),
                    diferencia_siguiente_z_mm=dz[j]*1000 if j<3 else None,
                    error_estimado_siguiente_z_mm=dz[j]*1000/15 if j<3 else None,
                    error_estimado_siguiente_theta_grados=float(np.degrees(dth[j])/15) if j<3 else None,
                    orden_observado_z=orden,regimen_orden4_verificado=regimen_4,
                    f_z_hz=metrics.get("f_z_hz"),rms_z_mm=metrics.get("rms_z_mm"),
                    nota="Estimación /15 condicionada a régimen de orden 4; dt/8 es referencia numérica, no exacta."
                         + (" Estos niveles no confirman todavía orden 4." if regimen_4 is False else ""))
        filas.append(fila)
    return filas


def barrido_velocidad(p, s, velocidades_kmh):
    filas = []
    for v in velocidades_kmh:
        caso = replace(s, perfil="constante", v_final_kmh=float(v))
        try:
            r = integrar(p, caso)
            _, met = diagnosticos(r,p,caso)
            lin = analisis_lineal(p,v/3.6)
            filas.append(dict(v_kmh=v, estado=met["estado"], rms_z_mm=met.get("rms_z_mm"),
                              semiamplitud_z_mm=met.get("semiamplitud_z_mm"), f_z_hz=met.get("f_z_hz"),
                              max_parte_real_s_inv=lin.get("max_parte_real_s_inv"),
                              notas="; ".join(met.get("notas",[]))))
        except (ValueError, RuntimeError) as exc:
            filas.append(dict(v_kmh=v, estado=str(exc)))
    return filas


def comparar_amortiguamiento(p,s):
    """Barrido propio: conserva el promedio de c por eje y varía el sesgo."""
    promedio_f = (p.cf_comp_ns_m+p.cf_ext_ns_m)/2
    promedio_r = (p.cr_comp_ns_m+p.cr_ext_ns_m)/2
    filas = []
    for razon in (1.0,1.5,2.0):
        modelo = replace(p,cf_comp_ns_m=2*promedio_f/(1+razon),
                         cf_ext_ns_m=2*razon*promedio_f/(1+razon),
                         cr_comp_ns_m=2*promedio_r/(1+razon),
                         cr_ext_ns_m=2*razon*promedio_r/(1+razon))
        r = integrar(modelo,s)
        _,met = diagnosticos(r,modelo,s)
        filas.append(dict(razon_ext_comp=razon,cf_comp_ns_m=modelo.cf_comp_ns_m,
                          cf_ext_ns_m=modelo.cf_ext_ns_m,cr_comp_ns_m=modelo.cr_comp_ns_m,
                          cr_ext_ns_m=modelo.cr_ext_ns_m,estado=r["estado"],
                          rms_z_mm=met.get("rms_z_mm"),f_z_hz=met.get("f_z_hz"),
                          semiamplitud_z_mm=met.get("semiamplitud_z_mm"),
                          nota="Barrido didáctico a promedio de c constante; no es optimización de F1."))
    return filas


# 7. Resultados reproducibles: CSV, configuración exacta y figuras sin redefinir el modelo.
def guardar_csv(path, filas):
    keys = list(dict.fromkeys(k for fila in filas for k in fila))
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(filas)


def graficar(r, data, p, s, carpeta):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.style.use(Path(__file__).with_name("estilo_cientifico.mplstyle"))
    t, x = r["t"], r["x"]
    captions = []
    def guardar(fig, nombre, caption):
        fig.savefig(carpeta/f"{nombre}.png", dpi=160)
        fig.savefig(carpeta/f"{nombre}.pdf")
        plt.close(fig)
        captions.append(f"**{nombre}.** {caption}")
    fig, axs = plt.subplots(3, 1, figsize=(10, 10), layout="constrained")
    series = [(data["velocidad"]*3.6, r"$V$ (km/h)"),
              (x[:,0]*1000, r"$z$ (mm)"), (np.degrees(x[:,2]), r"$\theta$ (°)")]
    for ax, (y,label), panel in zip(axs,series,"abc"):
        ax.plot(t,y,"k-")
        ax.set_ylabel(label)
        ax.text(0.01,0.92,f"({panel})",transform=ax.transAxes)
    axs[-1].set_xlabel(r"$t$ (s)")
    guardar(fig,"Fig1_respuesta",f"Escenario ilustrativo, {r['metodo']}, Δt={r['dt']:.6g} s. (a) Velocidad prescrita; (b) coordenada vertical; (c) cabeceo positivo hacia abajo del frente. Estado: {r['estado']}.")
    fig, axs = plt.subplots(2,1,figsize=(10,8),layout="constrained")
    for key,style,label in [("piso","k-","Piso"),("frente","k--","Frente"),("trasero","k:","Trasero")]:
        axs[0].plot(t,data[key]/1000,style,label=label)
    axs[0].set_ylabel(r"$D$ (kN)")
    axs[0].legend()
    axs[1].plot(t,data["acc_rhs"],"k-",label="Ecuaciones")
    axs[1].plot(t,data["acc_df"],"k--",label="Diferencias finitas")
    axs[1].set_ylabel(r"$\ddot z$ (m/s²)")
    axs[1].set_xlabel(r"$t$ (s)")
    axs[1].legend()
    for ax,panel in zip(axs,"ab"):
        ax.text(0.01,0.92,f"({panel})",transform=ax.transAxes)
    guardar(fig,"Fig2_fuerzas_aceleracion","(a) Magnitudes de carga aerodinámica hacia abajo; (b) aceleración de RHS y segunda diferencia de z. Todas las curvas usan los mismos parámetros.")
    if "espectro_z" in data:
        fig, axs = plt.subplots(2,1,figsize=(10,8),layout="constrained")
        for ax, key, scale, label, panel in [(axs[0],"espectro_z",1000,r"$A_z$ (mm)","a"),
                                           (axs[1],"espectro_theta",180/np.pi,r"$A_\theta$ (°)","b")]:
            e = data[key]
            ax.plot(e["f"], e["amplitud"]*scale,"k-")
            ax.set_xlim(0,min(20,e["nyquist_hz"]))
            ax.set_ylabel(label)
            ax.text(0.01,0.92,f"({panel})",transform=ax.transAxes)
        axs[-1].set_xlabel(r"$f$ (Hz)")
        guardar(fig,"Fig3_espectros",f"(a) z; (b) theta. Ventana t≥{s.analizar_desde_s:g} s, tendencia lineal retirada, Hann y corrección de ganancia coherente. No prueba por sí sola oscilación sostenida.")
    fig, axs = plt.subplots(2,1,figsize=(10,8),layout="constrained")
    h = np.linspace(max(p.h_contacto_m*1.1,p.h_stall_m-10*p.ancho_stall_m),p.h_aero_ref_m*1.2,1000)
    axs[0].plot(h*1000,0.5*p.rho_kg_m3*(70.0**2)*p.cla_piso_m2*factor_piso(h,p)/1000,"k-")
    axs[0].set_xlabel(r"$h$ (mm)")
    axs[0].set_ylabel(r"$D_{piso}$ (kN)")
    u = np.linspace(-0.5,0.5,1001)
    for comp,ext,style,label in [(p.cf_comp_ns_m,p.cf_ext_ns_m,"k-","Frente"),
                               (p.cr_comp_ns_m,p.cr_ext_ns_m,"k--","Trasero")]:
        axs[1].plot(u,-coef_amortiguador(u,comp,ext,p.suavizado_amortiguador_m_s)*u/1000,style,label=label)
    axs[1].set_xlabel(r"$u$ (m/s)")
    axs[1].set_ylabel(r"$F_d$ (kN)")
    axs[1].legend()
    for ax,panel in zip(axs,"ab"):
        ax.text(0.01,0.92,f"({panel})",transform=ax.transAxes)
    guardar(fig,"Fig4_constitutivas","(a) Carga estática del piso a V=70 m/s; (b) fuerzas de amortiguador equivalentes por eje: u<0 compresión, u>0 extensión. Valores ilustrativos; frente y trasero pueden coincidir.")
    fig, axs = plt.subplots(2,1,figsize=(10,8),layout="constrained")
    axs[0].plot(t,data["potencia_d"],"k-",label=r"$P_d$")
    axs[0].plot(t,data["potencia_a"],"k--",label=r"$P_a$")
    axs[0].set_ylabel(r"$P$ (W)")
    axs[0].legend()
    axs[1].plot(t,data["error_energia"],"k-")
    axs[1].set_ylabel(r"$E-E_0-W$ (J)")
    axs[1].set_xlabel(r"$t$ (s)")
    for ax,panel in zip(axs,"ab"):
        ax.text(0.01,0.92,f"({panel})",transform=ax.transAxes)
    guardar(fig,"Fig5_balance","(a) Potencia disipada por amortiguadores y trabajo aerodinámico sobre movimientos vertical/cabeceo; (b) residual de energía mecánica. W se integra con trapecios, de orden 2: este residual no mide exclusivamente el error de RK4.")
    (carpeta/"CAPTIONS.md").write_text("# Figuras del escenario de ejemplo\n\n"+"\n\n".join(captions)+"\n",encoding="utf-8")


def graficar_convergencia(filas,carpeta):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    validas = [r for r in filas if r.get("diferencia_siguiente_z_mm") and r["diferencia_siguiente_z_mm"]>0]
    if len(validas)<2:
        return
    plt.style.use(Path(__file__).with_name("estilo_cientifico.mplstyle"))
    dt = np.array([r["dt_s"] for r in validas])
    error = np.array([r["diferencia_siguiente_z_mm"] for r in validas])
    fig,ax = plt.subplots(figsize=(9,6),layout="constrained")
    ax.loglog(dt,error,"ko-",markerfacecolor="white",label="Diferencia entre mallas")
    ax.loglog(dt,error[0]*(dt/dt[0])**4,"k--",label=r"$\propto\Delta t^4$")
    ax.set_xlabel(r"$\Delta t$ (s)")
    ax.set_ylabel(r"$\|z_{\Delta t}-z_{\Delta t/2}\|_\infty$ (mm)")
    ax.legend()
    for ext in ("png","pdf"):
        fig.savefig(carpeta/f"Fig6_convergencia.{ext}",dpi=160)
    plt.close(fig)
    with (carpeta/"CAPTIONS.md").open("a",encoding="utf-8") as file:
        file.write("\n\n**Fig6_convergencia.** Diferencia de z entre mallas RK4 anidadas evaluada en los mismos tiempos. La línea discontinua indica una pendiente 4 de referencia; no se fuerza esa pendiente en los resultados.\n")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config",type=Path,default=Path(__file__).with_name("parametros_ejemplo.json"))
    parser.add_argument("--salida",type=Path,default=Path(__file__).with_name("resultados"),help="Carpeta de CSV, JSON y figuras")
    parser.add_argument("--perfil",choices=["rampa","constante"])
    parser.add_argument("--velocidad",type=float,help="Velocidad objetivo en km/h")
    parser.add_argument("--dt",type=float)
    parser.add_argument("--tau-aero",type=float)
    parser.add_argument("--convergencia",action="store_true")
    parser.add_argument("--barrido",action="store_true")
    parser.add_argument("--comparar-amortiguamiento",action="store_true")
    parser.add_argument("--sin-figuras",action="store_true")
    args = parser.parse_args()
    cfg = json.loads(args.config.read_text(encoding="utf-8"))
    p, s = Parametros(**cfg["modelo"]), Simulacion(**cfg["simulacion"])
    if args.tau_aero is not None:
        p = replace(p,tau_aero_s=args.tau_aero)
    overrides = {key:value for key,value in [("perfil",args.perfil),("v_final_kmh",args.velocidad),("dt_s",args.dt)] if value is not None}
    s = replace(s,**overrides)
    r = integrar(p,s)
    data, summary = diagnosticos(r,p,s)
    args.salida.mkdir(parents=True,exist_ok=True)
    (args.salida/"configuracion_usada.json").write_text(json.dumps(dict(modelo=asdict(p),simulacion=asdict(s)),indent=2,ensure_ascii=False),encoding="utf-8")
    summary["analisis_lineal_inicial"] = analisis_lineal(p,float(velocidad(0,s)))
    (args.salida/"resumen.json").write_text(json.dumps(summary,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
    rows = []
    for i,(t,x) in enumerate(zip(r["t"],r["x"])):
        row = dict(t_s=t,z_m=x[0],dz_m_s=x[1],theta_rad=x[2],dtheta_rad_s=x[3])
        unidades = {"velocidad":"velocidad_m_s","acc_rhs":"acc_rhs_m_s2","acc_df":"acc_df_m_s2",
                    "piso":"piso_n","frente":"frente_n","trasero":"trasero_n","cf":"cf_ns_m",
                    "cr":"cr_ns_m","potencia_d":"potencia_d_w","potencia_a":"potencia_a_w",
                    "error_energia":"error_energia_j","zf":"zf_m","zr":"zr_m","uf":"uf_m_s","ur":"ur_m_s"}
        for key,columna in unidades.items():
            row[columna] = data[key][i]
        row["h_piso_cp_m"] = x[0]-p.cg_a_piso_m+p.d_piso_m*x[2]
        rows.append(row)
    guardar_csv(args.salida/"trayectoria.csv",rows)
    if not args.sin_figuras:
        graficar(r,data,p,s,args.salida)
    if args.convergencia:
        filas = estudiar_convergencia(p,s)
        guardar_csv(args.salida/"convergencia.csv",filas)
        if not args.sin_figuras:
            graficar_convergencia(filas,args.salida)
    if args.barrido:
        guardar_csv(args.salida/"barrido_velocidad.csv",barrido_velocidad(p,s,[0,120,180,240,300,330]))
    if args.comparar_amortiguamiento:
        guardar_csv(args.salida/"comparacion_amortiguamiento.csv",comparar_amortiguamiento(p,s))
    print(json.dumps(summary,indent=2,ensure_ascii=False))
    print(f"Archivos guardados en: {args.salida.resolve()}")


if __name__ == "__main__" and "__file__" in globals():
    main()
